In [6]:
import os
import shutil
from pyspark.sql import SparkSession
# ... other imports ...

# --- Configuration ---
ICEBERG_VERSION = "1.5.0"
LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
CATALOG_NAME = "local"

# --- Stop existing SparkSession ---
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception:
    pass

# # --- Clean warehouse ---
# # Note: For production use, should not be cleaned the warehouse every time!
# if os.path.exists(LOCAL_WAREHOUSE_PATH):
#     print(f"Cleaning up old warehouse: {LOCAL_WAREHOUSE_PATH}")
#     # Using os.path.join for robust path handling on Windows
#     # The f"file://{LOCAL_WAREHOUSE_PATH}" is for the Spark config, not os.path.exists
#     try:
#         shutil.rmtree(LOCAL_WAREHOUSE_PATH)
#     except Exception as e:
#         print(f"Warning: Could not fully clean path. Please ensure no files are open. Error: {e}")


# --- Iceberg packages ---
ICEBERG_PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION},"
    f"org.apache.avro:avro:1.11.3"
)

## --- SparkSession ---
spark = SparkSession.builder \
    .appName("IcebergDescribeExample") \
    .config("spark.jars.packages", ICEBERG_PACKAGES) \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "hadoop") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.python.authenticate.socketTimeout", "120s")\
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .config("spark.rpc.message.maxSize", "512")\
    .config("spark.storage.blockManagerTimeoutIntervalMs", "300000")\
    \
    .getOrCreate()

print("Spark version:", spark.version)
print(f"Spark Session and Iceberg Warehouse is set to: {spark.conf.get(f'spark.sql.catalog.{CATALOG_NAME}.warehouse')}")


Spark version: 3.5.7
Spark Session and Iceberg Warehouse is set to: file:////data/data_files/iceberg/iceberg_warehouse


In [23]:
spark = SparkSession.builder.appName("DataFrameCreation").getOrCreate()

In [24]:
import pyspark.sql.functions as sf
from pyspark.sql import SparkSession
data = [("Alice", 1, "New York"), ("Bob", 2, "London"), ("Charlie", 3, "Paris")]
columns = ["Name", "ID", "City"]
df = spark.createDataFrame(data, schema=columns)
df.show()
# df=spark.sql("select sequence(1, 9) A")
# df.select(sf.explode('a')).show()

# df = spark.createDataFrame([(-2, 2)], ['start', 'stop'])
# df.show()
# # df.select(sf.sequence(df.start, df.stop)).show()

Py4JJavaError: An error occurred while calling o446.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (192.168.1.12 executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:203)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:331)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(

In [2]:
df_Sales_SalesOrderHeader = spark.read.orc("C:/data/data_files/orc/Sales/SalesOrderHeader")
df_Sales_Customer = spark.read.orc("C:/data/data_files/orc/Sales/Customer/")
df_Person_Person = spark.read.orc("C:/data/data_files/orc/Person/Person/")
df_Sales_SalesOrderDetail = spark.read.orc("C:/data/data_files/orc/Sales/SalesOrderDetail/")
df_Production_Product = spark.read.orc("C:/data/data_files/orc/Production/Product/")

df_Person_Person.writeTo("local.Person_ORC.Person").createOrReplace()
df_Sales_Customer.writeTo("local.Sales_ORC.Customer").createOrReplace()
df_Sales_SalesOrderHeader.writeTo("local.Sales_ORC.SalesOrderHeader").createOrReplace()
df_Sales_SalesOrderDetail.writeTo("local.Sales_ORC.SalesOrderDetail").createOrReplace()
df_Production_Product.writeTo("local.Production_ORC.Product").createOrReplace()


In [5]:
print("C:/data/data_files/orc/Sales/SalesOrderHeader.orc = " + str(df_Sales_SalesOrderHeader.count()))
print("C:/data/data_files/orc/Sales/Customer.orc = " + str(df_Sales_Customer.count()))
print("C:/data/data_files/orc/Person/Person.orc = " + str(df_Person_Person.count()))
print("C:/data/data_files/orc/Sales/SalesOrderDetail.orc = " + str(df_Sales_SalesOrderDetail.count()))
print("C:/data/data_files/orc/Production/Product.orc = " + str(df_Production_Product.count()))
print("local.Sales.SalesOrderHeader = " + str(spark.table("local.Sales.SalesOrderHeader").count()))
print("local.Sales.Customer = " + str(spark.table("local.Sales.Customer").count()))
print("local.Person.Person = " + str(spark.table("local.Person.Person").count()))
print("local.Sales.SalesOrderDetail = " + str(spark.table("local.Sales.SalesOrderDetail").count()))
print("local.Production.Product = " + str(spark.table("local.Production.Product").count()))

C:/data/data_files/orc/Sales/SalesOrderHeader.orc = 31465
C:/data/data_files/orc/Sales/Customer.orc = 19820
C:/data/data_files/orc/Person/Person.orc = 19972
C:/data/data_files/orc/Sales/SalesOrderDetail.orc = 121317
C:/data/data_files/orc/Production/Product.orc = 504
local.Sales.SalesOrderHeader = 31465
local.Sales.Customer = 19820
local.Person.Person = 19972
local.Sales.SalesOrderDetail = 121317
local.Production.Product = 504


In [7]:
from pyspark.sql import functions as F

soh = spark.table("local.Sales.SalesOrderHeader").alias("soh")
c = spark.table("local.Sales.Customer").alias("c")
p = spark.table("local.Person.Person").alias("p")
sod = spark.table("local.Sales.SalesOrderDetail").alias("sod")
prd = spark.table("local.Production.Product").alias("prd")


joined_df = soh.join(
    c,
    soh["CustomerID"] == c["CustomerID"],
    how="inner"
) \
.join(
    p, c["PersonID"] == p["BusinessEntityID"],
    how="inner"
) \
.join(
    sod, soh["SalesOrderID"] == sod["SalesOrderID"],
    how="inner"
) \
.join(
    prd, sod["ProductID"] == prd["ProductID"],
    how="inner"
) \
.select(
    F.col("soh.SalesOrderID"),
    F.col("soh.OrderDate"),
    F.col("soh.DueDate"),
    F.col("soh.ShipDate"),
    F.col("soh.Status"),
    F.col("soh.OnlineOrderFlag"),
    F.col("soh.SalesOrderNumber"),
    F.col("soh.PurchaseOrderNumber"),
    F.col("soh.SubTotal"),
    F.col("soh.TaxAmt"),
    F.col("soh.Freight"),
    F.col("soh.TotalDue"),
    F.col("soh.Comment"),
    F.col("c.CustomerID"),
    F.col("p.firstName"),
    F.col("p.lastName"),
    F.col("c.AccountNumber"),
    F.col("prd.name").alias("ProductName"),
    F.col("sod.OrderQty"),
    F.col("sod.UnitPrice"),
    F.col("sod.LineTotal"),
    F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy"))
)

# joined_df.show(5, truncate=False)
mayFilter = joined_df.filter((F.year(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 5) & (F.col("AccountNumber") == "AW00029825"))

mayFilter.show(5, truncate=False)


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\data\python\Python312\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\data\python\Python312\Lib\socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\data\python\Python312\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\data\python\Python312\Lib\site-packages\py4j\clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while s

Py4JError: An error occurred while calling o217.showString

In [ ]:
df_orcPerson = spark.table("local.Person_ORC.Person").alias("P")
df_orcPerson.show(10, False)

In [6]:
from pyspark.sql import functions as F

soh = df_Sales_SalesOrderHeader.alias("soh")
c = df_Sales_Customer.alias("c")
p = df_Person_Person.alias("p")
sod = df_Sales_SalesOrderDetail.alias("sod")
prd = df_Production_Product.alias("prd")

joined_df = soh.join(
    c,
    soh["CustomerID"] == c["CustomerID"],
    how="inner"
) \
.join(
    p, c["PersonID"] == p["BusinessEntityID"],
    how="inner"
) \
.join(
    sod, soh["SalesOrderID"] == sod["SalesOrderID"],
    how="inner"
) \
.join(
    prd, sod["ProductID"] == prd["ProductID"],
    how="inner"
) \
.select(
    F.col("soh.SalesOrderID"),
    F.col("soh.OrderDate"),
    F.col("soh.DueDate"),
    F.col("soh.ShipDate"),
    F.col("soh.Status"),
    F.col("soh.OnlineOrderFlag"),
    F.col("soh.SalesOrderNumber"),
    F.col("soh.PurchaseOrderNumber"),
    F.col("soh.SubTotal"),
    F.col("soh.TaxAmt"),
    F.col("soh.Freight"),
    F.col("soh.TotalDue"),
    F.col("soh.Comment"),
    F.col("c.CustomerID"),
    F.col("p.firstName"),
    F.col("p.lastName"),
    F.col("c.AccountNumber"),
    F.col("prd.name").alias("ProductName"),
    F.col("sod.OrderQty"),
    F.col("sod.UnitPrice"),
    F.col("sod.LineTotal"),
    F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy"))
)

# joined_df.show(5, truncate=False)
mayFilter = joined_df.filter((F.year(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 5) & (F.col("AccountNumber") == "AW00029825"))

mayFilter.show(5, truncate=False)


+------------+---------+---------+--------+------+---------------+----------------+-------------------+----------+---------+--------+----------+-------+----------+---------+----------+-------------+---------------------------+--------+---------+---------+-----------------------------------+
|SalesOrderID|OrderDate|DueDate  |ShipDate|Status|OnlineOrderFlag|SalesOrderNumber|PurchaseOrderNumber|SubTotal  |TaxAmt   |Freight |TotalDue  |Comment|CustomerID|firstName|lastName  |AccountNumber|ProductName                |OrderQty|UnitPrice|LineTotal|month(to_date(OrderDate, M/d/yyyy))|
+------------+---------+---------+--------+------+---------------+----------------+-------------------+----------+---------+--------+----------+-------+----------+---------+----------+-------------+---------------------------+--------+---------+---------+-----------------------------------+
|7/13/2019   |5/31/2011|6/12/2011|6/7/2011|5     |0              |SO43659         |PO522145787        |20565.6206|1971.5149|

In [ ]:
from pyspark.sql import SparkSession
# spark.sql("show catalogs").show()
spark.sql("show databases in local").show()
# spark.sql("show tables in local").show()
# # spark.catalog
# # spark.conf.get("catalog")
# spark.catalog.listCatalogs()
# spark.catalog.listDatabases(local)


In [ ]:
spark.stop()

In [ ]:
import pyspark.sql.functions as sf
# spark.sql("Select EXPLODE(sequence(1, 9)) nos").show()
# df = spark.createDataFrame([(1)], ['col'])
# df.show()
# df.select(sf.lit(sf.sequence(1, 9))).show()
spark.range(1).select(sf.current_schema()).show()

In [ ]:
# spark.catalog.setCurrentCatalog("local")
# # spark.catalog.currentCatalog()
# # spark.catalog.listCatalogs()
# # spark.catalog.setCurrentDatabase("manual")
# # spark.catalog.listDatabases()
# # spark.catalog.listTables()

In [ ]:
spark.catalog.createTable("tbl_test", schema=spark.range(1).schema, source='parquet')

In [ ]:
from pyspark.sql import SparkSession
# spark.catalog.listColumns("BusinessEntityContact")
spark.catalog.getTable("tbl_test")
# spark.sql("DESCRIBE EXTENDED tbl_test").show(truncate=False)

In [ ]:
# spark.sql("Show databases in local").show()
spark.sql("Show tables in manual_Person").show()
# spark.sql("Show tables in manual_Sales").show()

In [ ]:
# dfsoh = spark.table("local.manual_Sales.SalesOrderHeader").alias("soh")
# dfsoh.printSchema()

In [ ]:
# # spark.sql("Show tables in sales").show()
# # dfCust = spark.table("local.Sales.Customer").alias("cust")
# # dfCust.printSchema()

# from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# schema = StructType([
#     StructField('CustomerID', IntegerType(), True), 
#     StructField('PersonID', IntegerType(), True), 
#     StructField('StoreID', IntegerType(), True), 
#     StructField('TerritoryID', IntegerType(), True), 
#     StructField('AccountNumber', StringType(), True), 
#     StructField('rowguid', StringType(), True), 
#     StructField('ModifiedDate', DateType(), True)]
#     )

# spark.catalog.createTable(
#     tableName="manual_Sales.Customer",
#     source="parquet",
#     schema=schema
# )

spark.sql("DESCRIBE EXTENDED manual_Sales.Customer").show(truncate=False)

In [ ]:
# spark.sql("Show tables in sales").show()
dfCust = spark.table("local.Sales.Customer").alias("cust")
# dfCust.count()
# dfCust.foreach
rows = dfCust.collect()
# for row in dfCust.toLocalIterator():
#     print(
#         row["CustomerID"],
#         row["PersonID"],
#         row["StoreID"],
#         row["TerritoryID"],
#         row["AccountNumber"],
#         row["rowguid"],
#         row["ModifiedDate"]
#     )
#     break

In [ ]:
# Step 1: Define SQL-to-PySpark type mapping

from pyspark.sql.types import *
import json


sql_to_pyspark_type = {
    "INT": IntegerType(),
    "INTEGER": IntegerType(),
    "BIGINT": LongType(),
    "SMALLINT": ShortType(),
    "TINYINT": ByteType(),
    "FLOAT": FloatType(),
    "DOUBLE": DoubleType(),
    "DECIMAL": DecimalType(10, 2),  # You can customize precision/scale
    "NUMERIC": DecimalType(10, 2),
    "CHAR": StringType(),
    "VARCHAR": StringType(),
    "TEXT": StringType(),
    "STRING": StringType(),
    "BOOLEAN": BooleanType(),
    "DATE": DateType(),
    "TIMESTAMP": TimestampType()
}

# Step 2: Create a parser function


def parse_sql_schema(sql_schema: str):
    fields = []
    for line in sql_schema.strip().split(","):
        name, dtype = line.strip().split()
        dtype_upper = dtype.upper()
        spark_type = sql_to_pyspark_type.get(dtype_upper)
        if spark_type is None:
            raise ValueError(f"Unsupported type: {dtype}")
        fields.append(StructField(name, spark_type, True))
    return StructType(fields)

# Example Usage

ddl = """
CustomerID INT,
PersonID INT,
StoreID INT,
TerritoryID INT,
AccountNumber STRING,
rowguid STRING,
ModifiedDate DATE
"""

schema = parse_sql_schema(ddl)
# schema.prettyJson()

print(schema)

# Convert StructType to JSON string
# schema_json = schema.json()

# Pretty-print it
# print(json.dumps(json.loads(schema_json), indent=2))


In [ ]:
from pyspark.sql.types import DataType

ddl_string = "b string, a int"
schema = DataType.fromDDL(ddl_string)
print(schema)

In [ ]:
from pyspark.sql.types import StructType
ddl = """
CustomerID INT,
PersonID INT,
StoreID INT,
TerritoryID INT,
AccountNumber STRING,
rowguid STRING,
ModifiedDate DATE
"""
schema = StructType.fromDDL(ddl)
schema